# This challenge adds another call to translate the Brochure to spanish.



In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from modified_scrapper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [2]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()


API key looks good so far


In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [5]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [6]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [7]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [8]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [9]:
def stream_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ]
    )
    result = response.choices[0].message.content

    stream = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": "Translate the following marketing brochure to Spanish: " + result}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [10]:
stream_brochure("OpenAI", "https://openai.com/")


Selecting relevant links for https://openai.com/ by calling gpt-5-nano
Found 17 relevant links


# OpenAI: Construyendo IA segura y beneficiosa para todos

 OpenAI es una empresa de investigación y despliegue de IA con la misión de garantizar que la inteligencia artificial general beneficia a toda la humanidad. Buscamos un futuro donde la IAG — sistemas de IA generalmente más inteligentes que los humanos — ayude a todos a prosperar.

## Nuestra Misión y Visión
- Nuestra misión: garantizar que la inteligencia artificial general beneficie a toda la humanidad.
- Nuestra visión: avanzar la IAG de forma segura y beneficiosa, asegurando al mismo tiempo que nuestro trabajo ayude a otros a lograr ese resultado.
- Nuestra Carta guía cómo perseguimos una IA segura y ampliamente beneficiosa.

## Cómo está Estructurada OpenAI
- OpenAI consta de dos entidades entrelazadas:
  - la Fundación OpenAI, sin fines de lucro
  - el Grupo OpenAI, con fines de lucro
- La Fundación gobierna al Grupo, que opera como una corporación de beneficio público.
- La Fundación posee acciones en OpenAI Group, con un valor actualmente estimado en aproximadamente 130 mil millones de dólares, lo que la convierte en una de las organizaciones filantrópicas mejor dotadas del mundo.

## Nuestro Trabajo: Investigación, Productos y Soluciones
- Investigación
  - OpenAI realiza investigación continua en múltiples dominios, incluidas las últimas avances como GPT-5.4, GPT-5.3 Instant y GPT-5.
  - Las áreas incluyen OpenAI para la Ciencia, Investigación Económica, y un amplio Índice de Investigación y programas de Residencias.
- Productos
  - Serie GPT (GPT-5.4, GPT-5.3 Instant, GPT-5, Codex, etc.)
  - ChatGPT para uso cotidiano y acceso para desarrolladores
  - Sora: visión general, características y precios
  - Codex: capacidades de IA centradas en código
  - Plataforma API: visión general de la plataforma, precios, documentación y recursos para desarrolladores
- Para Empresas y Desarrolladores
  - Soluciones empresariales, ofertas enfocadas en educación y IA escalable como servicio
  - Recursos para desarrolladores: Documentación, Foro de Desarrolladores y acceso a API
- Actualizaciones más recientes
  - Noticias sobre nuevas formas de aprender matemáticas y ciencias con ChatGPT, vistazos de seguridad como Codex Security y alianzas estratégicas

## Asociaciones, Clientes y Impacto en el Mundo Real
- Asociaciones estratégicas y colaboraciones
  - OpenAI y Amazon anuncian una asociación estratégica
  - Declaración Conjunta de OpenAI y Microsoft
  - Entorno de Ejecución con Estado para Agentes en Amazon Bedrock
- Gobierno e industria
  - Acuerdo de OpenAI con el Departamento de Guerra
- Historias de clientes y casos de uso
  - Taisei Corporation da forma a la próxima generación de talento con ChatGPT
  - Historias en diversos sectores muestran IA ayudando a operaciones y aprendizaje (ejemplos: una chatarrería, una granja de semillas y una tienda de tamales)
- Estas asociaciones e historias ilustran cómo se aplica la tecnología de OpenAI a través de industrias y geografías

## OpenAI para Empresas y Desarrolladores
- Para Negocios: soluciones personalizadas para empresas y educación
- Plataforma API: herramientas para que los desarrolladores integren capacidades de IA en sus propias aplicaciones
- Recursos y precios para adaptarse a organizaciones de distintos tamaños

## Cultura y Carreras
- Equipos diversos e interdisciplinarios: Desarrollar IA segura y beneficiosa requiere personas de una amplia variedad de disciplinas y orígenes.
- Nuestras oportunidades de carrera: OpenAI está contratando activamente en roles y disciplinas para dar forma al futuro de la tecnología.
- Carreras en OpenAI: Explora oportunidades y únete a nosotros para construir IA responsable.

## Participa y Empieza a Explorar
- Prueba ChatGPT para uso personal y exploración
- Comienza con ChatGPT para desarrolladores y empresas
- Explora nuestra investigación y mantente al tanto de los últimos avances
- Infórmate sobre la Fundación OpenAI y nuestra gobernanza, incluida nuestra Carta y estructura

OpenAI está comprometida con el liderazgo en IA general segura y beneficiosa, respaldada por una base sólida, alianzas activas y una cultura que valora talentos diversos y una responsabilidad rigurosa. Si tienes curiosidad sobre cómo la IA puede transformar tu campo u organización, ya sea como usuario, desarrollador, cliente o posible compañero de equipo, OpenAI ofrece formas de involucrarte y colaborar.